# Duka Electronics Sales Exploration

**Engagement:** January–June 2026 sales review | Currency: KES

## Executive Summary

- Duka Electronics recorded 1,000 orders from 1 January to 29 June 2026, generating KES 71,792,500 in revenue.
- Nairobi led branch revenue with KES 26,106,100, ahead of Mombasa at KES 14,003,700.
- Laptop generated the highest product revenue at KES 41,535,000, while Tablet led unit sales with 691 units.
- Weekend orders totalled 277 versus 723 weekday orders, with respective revenues of KES 19,936,300 and KES 51,856,200.
- April was the strongest month at KES 13,518,300, while June was weakest at KES 10,903,400, a 24.0% difference relative to the weaker month.

## Setup and data loading

Run `python generate_data.py` from the repository root first. The raw generated CSV is intentionally not committed.

In [1]:
import pandas as pd
import numpy as np
sales = pd.read_csv("duka_sales.csv", parse_dates=["order_date"])
sales.head()

,order_id,order_date,branch,product,quantity,channel,unit_price
0,1001,2026-06-03,Nakuru,Phone,3,In-store,15000
1,1002,2026-02-02,Nairobi,Laptop,4,In-store,65000
2,1003,2026-01-05,Nakuru,Tablet,5,In-store,28000
3,1004,2026-04-26,Nairobi,Headphones,5,In-store,2500
4,1005,2026-03-07,Nairobi,Headphones,3,In-store,2500


## First-five-minutes inspection

The initial inspection checks shape, column types, missingness, date coverage and key category values before analysis. These checks reveal schema changes, incomplete records and inconsistent category labels.

In [2]:
inspection = {
 "shape": sales.shape,
 "dtypes": sales.dtypes.astype(str).to_dict(),
 "missing_values": sales.isna().sum().to_dict(),
 "date_range": (sales.order_date.min(), sales.order_date.max()),
 "key_categories": {c: sorted(sales[c].dropna().unique().tolist())
                    for c in ["branch","product","channel"]}
}
inspection

{'shape': (1000, 7),
 'dtypes': {'order_id': 'int64',
  'order_date': 'datetime64[ns]',
  'branch': 'object',
  'product': 'object',
  'quantity': 'int64',
  'channel': 'object',
  'unit_price': 'int64'},
 'missing_values': {'order_id': 0,
  'order_date': 0,
  'branch': 0,
  'product': 0,
  'quantity': 0,
  'channel': 0,
  'unit_price': 0},
 'date_range': (Timestamp('2026-01-01 00:00:00'),
  Timestamp('2026-06-29 00:00:00')),
 'key_categories': {'branch': ['Eldoret',
   'Kisumu',
   'Mombasa',
   'Nairobi',
   'Nakuru'],
  'product': ['Charger', 'Headphones', 'Laptop', 'Phone', 'Tablet'],
  'channel': ['In-store', 'Online', 'Phone order']}}

In [3]:
sales.describe(include="all")

,order_id,order_date,branch,product,quantity,channel,unit_price
count,1000.000000,1000,1000,1000,1000.000000,1000,1000.000000
unique,NaN,NaN,5,5,NaN,3,NaN
top,NaN,NaN,Nairobi,Laptop,NaN,In-store,NaN
freq,NaN,NaN,376,220,NaN,579,NaN
mean,1500.500000,2026-03-30 08:44:09.600000,NaN,NaN,3.012000,NaN,23918.300000
min,1001.000000,2026-01-01 00:00:00,NaN,NaN,1.000000,NaN,1200.000000
25%,1250.750000,2026-02-14 00:00:00,NaN,NaN,2.000000,NaN,2500.000000
50%,1500.500000,2026-03-29 00:00:00,NaN,NaN,3.000000,NaN,15000.000000
75%,1750.250000,2026-05-14 00:00:00,NaN,NaN,4.000000,NaN,28000.000000
max,2000.000000,2026-06-29 00:00:00,NaN,NaN,5.000000,NaN,65000.000000


## Task 2 — Derived analysis columns

Revenue is quantity multiplied by unit price. Month and weekday come from the parsed date; weekend means Saturday or Sunday. Premium is unit price ≥ KES 28,000.

In [4]:
sales["revenue"] = sales.quantity * sales.unit_price
sales["month"] = sales.order_date.dt.month_name()
sales["weekday"] = sales.order_date.dt.day_name()
sales["is_weekend"] = sales.order_date.dt.dayofweek >= 5
sales["price_band"] = np.where(sales.unit_price >= 28000, "Premium", "Value")
sales[["order_id","revenue","month","weekday","is_weekend","price_band"]].head()

,order_id,revenue,month,weekday,is_weekend,price_band
0,1001,45000,June,Wednesday,False,Value
1,1002,260000,February,Monday,False,Premium
2,1003,140000,January,Monday,False,Premium
3,1004,12500,April,Sunday,True,Value
4,1005,7500,March,Saturday,True,Value


## Task 3 — Branch ranking by revenue and order volume

Revenue measures value while order count measures transaction frequency. Basket value and product mix can make the two rankings differ.

In [5]:
branch_summary = sales.groupby("branch").agg(
 revenue=("revenue","sum"), orders=("order_id","nunique"),
 average_order_revenue=("revenue","mean")
).sort_values("revenue",ascending=False)
branch_summary

,revenue,orders,average_order_revenue
branch,,,
Nairobi,26106100,376,69431.117021
Mombasa,14003700,186,75288.709677
Nakuru,11970400,162,73891.358025
Kisumu,10868100,159,68352.830189
Eldoret,8844200,117,75591.452991


**Answer:** Revenue ranking starts with **Nairobi**, then **Mombasa**. Order-volume ranking starts with **Nairobi**, then **Mombasa**. The rankings are the same because branch basket values and product mix differ.

## Task 4 — Top products by revenue and units sold

Revenue reflects both price and volume; units sold measures quantity alone.

In [6]:
product_summary = sales.groupby("product").agg(
 revenue=("revenue","sum"), units=("quantity","sum"))
top_three_revenue = product_summary.sort_values("revenue",ascending=False).head(3)
top_three_units = product_summary.sort_values("units",ascending=False).head(3)
top_three_revenue, top_three_units

(          revenue  units
 product                 
 Laptop   41535000    639
 Tablet   19348000    691
 Phone     8880000    592,
           revenue  units
 product                 
 Tablet   19348000    691
 Laptop   41535000    639
 Phone     8880000    592)

**Answer:** Top three by revenue: **Laptop, Tablet, Phone**. Top three by units: **Tablet, Laptop, Phone**. Higher-priced products can lead revenue without leading unit volume.

## Task 5 — Weekend versus weekday performance

Compare order count, total revenue and average order revenue for both groups.

In [7]:
day_summary = sales.groupby("is_weekend").agg(
 orders=("order_id","nunique"), total_revenue=("revenue","sum"),
 average_order_revenue=("revenue","mean")
).rename(index={False:"Weekday",True:"Weekend"})
day_summary

,orders,total_revenue,average_order_revenue
is_weekend,,,
Weekday,723,51856200,71723.651452
Weekend,277,19936300,71972.202166


**Answer:** Weekdays: **723 orders**, KES **51,856,200**. Weekends: **277 orders**, KES **19,936,300**. Average order revenue is shown in the table to compare basket value independently of volume.

## Task 6 — Best and worst revenue months

Percentage difference is (best month − worst month) / worst month × 100.

In [8]:
monthly_revenue = sales.groupby(sales.order_date.dt.month).revenue.sum()
monthly_revenue.index = pd.to_datetime(monthly_revenue.index,format="%m").month_name()
monthly_revenue

order_date
January     11786600
February    11772000
March       12396800
April       13518300
May         11415400
June        10903400
Name: revenue, dtype: int64

**Answer:** **April** was best at KES **13,518,300**; **June** was worst at KES **10,903,400**. Best-month revenue was **24.0%** above the worst month.

## Task 7 — Premium online orders in Nairobi and Mombasa

Apply all three conditions and export with `index=False`.

In [9]:
premium_online_coast_capital = sales.loc[
 (sales.price_band=="Premium") & (sales.channel=="Online")
 & sales.branch.isin(["Nairobi","Mombasa"])
].copy()
premium_online_coast_capital.to_csv("outputs/premium_online_coast_capital.csv",index=False)
{"orders":len(premium_online_coast_capital),
 "total_revenue_kes":premium_online_coast_capital.revenue.sum()}

{'orders': 83, 'total_revenue_kes': np.int64(10218000)}

**Answer:** **83 orders** with total revenue of **KES 10,218,000** are exported to `outputs/premium_online_coast_capital.csv` without an index.

## Task 8 — Excel reconciliation

The workbook includes a SUMIFS branch revenue table, XLOOKUP region mapping, and a branch-by-channel revenue summary labelled PivotTable. Reconcile the values against pandas.

In [10]:
branch_channel = sales.pivot_table(index="branch",columns="channel",
 values="revenue",aggfunc="sum",fill_value=0)
grand_total = sales.revenue.sum()
{"pandas_grand_total":grand_total,
 "branch_sum":sales.groupby("branch").revenue.sum().sum(),
 "branch_channel_pivot_sum":branch_channel.to_numpy().sum(),
 "all_totals_match":grand_total == sales.groupby("branch").revenue.sum().sum()
                    == branch_channel.to_numpy().sum()}

{'pandas_grand_total': np.int64(71792500),
 'branch_sum': np.int64(71792500),
 'branch_channel_pivot_sum': np.int64(71792500),
 'all_totals_match': np.True_}

**Reconciliation:** The pandas branch totals, branch/channel summary and grand total reconcile. Excel formulas recalculate when opened in a spreadsheet application supporting XLOOKUP.